<a href="https://colab.research.google.com/github/theusualsuspects/iitk-examples/blob/main/Live_Hands_on.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers faiss-cpu groq langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.7 MB/s eta 0:00:00


In [ ]:
sales_records = [
    "South region revenue fell 12% in September 2026.",
    "North region revenue grew 5% in September 2026.",
    "East region revenue was flat in September 2026.",
]

code_commits = [
    "Sept 3 commit: changed the discount rule in the pricing engine.",
    "Aug 20 commit: fixed a checkout bug unrelated to pricing.",
    "Sept 10 commit: refactored the logging module.",
]

policy_docs = [
    "Policy memo: new 15% discount cap approved by finance on Sept 2.",
    "Policy memo: vacation policy updated in July.",
    "Compliance doc: data retention policy unchanged since 2024.",
]

SOURCES = {"sales": sales_records, "code": code_commits, "policy": policy_docs}

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

indexes = {}
for name, records in SOURCES.items():
    embs = embedder.encode(records, normalize_embeddings=True).astype("float32")
    idx = faiss.IndexFlatIP(embs.shape[1])   # normalized vectors -> inner product = cosine similarity
    idx.add(embs)
    indexes[name] = idx

def search(source, query):
    q = embedder.encode([query], normalize_embeddings=True).astype("float32")
    _, top = indexes[source].search(q, 1)
    return SOURCES[source][top[0][0]]

In [ ]:
FUNCS = {
    "search_sales":  lambda query: search("sales", query),
    "search_code":   lambda query: search("code", query),
    "search_policy": lambda query: search("policy", query),
}

TOOL_SCHEMAS = [
    {"type": "function", "function": {
        "name": name, "description": desc,
        "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}}
    for name, desc in [
        ("search_sales",  "Search sales and revenue records by region/month."),
        ("search_code",   "Search recent code commits and changes."),
        ("search_policy", "Search policy memos and approval documents."),
    ]
]

In [ ]:
import os, getpass
from groq import Groq

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")
client = Groq()
MODEL = "openai/gpt-oss-20b"